# 11 — Mississippi Alluvial Plain, SPOT 6/7 with Delineate-Anything

Delineates agricultural fields in the Mississippi Alluvial Plain (MAP) using **SPOT 6/7** imagery at 6 m resolution and the **Delineate-Anything** engine. Includes cross-year stability analysis using IoU/F1.

> **Note:** SPOT 6/7 GEE access is restricted to select users (internal DRI use only). External users can contact the agribound author (sayantan.majumdar@dri.edu) to request processing.

**Estimated runtime:** ~15–30 minutes (3 years, 6 m resolution, GPU).

**Prerequisites:**
```bash
pip install agribound[gee,delineate-anything]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
from pathlib import Path

import agribound
from agribound.evaluate import evaluate

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

OUTPUT_DIR = Path("../../outputs/mississippi_alluvial_plain")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE = "spot"
ENGINE = "delineate-anything"
YEARS = [2021, 2022, 2023]

## Create Study Area

AOI in the central MAP near Greenville, Mississippi — dominated by row crops (cotton, soybeans, rice, corn).

In [ ]:
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [-91.10, 33.30],
                        [-90.90, 33.30],
                        [-90.90, 33.45],
                        [-91.10, 33.45],
                        [-91.10, 33.30],
                    ]
                ],
            },
            "properties": {"name": "Mississippi Alluvial Plain AOI"},
        }
    ],
}
study_area = str(OUTPUT_DIR / "map_aoi.geojson")
with open(study_area, "w") as f:
    json.dump(aoi, f)

print(f"Study area: {study_area}")

## Run Delineation (2021–2023)

In [ ]:
all_results = {}

for year in YEARS:
    print(f"Processing {year} — SPOT 6/7 at 6 m resolution...", end=" ")
    output_path = OUTPUT_DIR / f"fields_spot_{year}.gpkg"

    try:
        gdf = agribound.delineate(
            study_area=study_area,
            source=SOURCE,
            year=year,
            engine=ENGINE,
            output_path=str(output_path),
            gee_project=GEE_PROJECT,
            cloud_cover_max=15,
            min_area=10000,
            simplify=3.0,
            device="auto",
        )
        all_results[year] = gdf
        print(f"{len(gdf)} fields")

        if "metrics:area" in gdf.columns:
            area_ha = gdf["metrics:area"].sum() / 10000
            avg_ha = gdf["metrics:area"].mean() / 10000
            print(f"  Total area: {area_ha:,.1f} ha, avg: {avg_ha:,.1f} ha")

    except Exception as exc:
        print(f"SPOT access error: {exc}")
        print(
            "SPOT 6/7 is restricted to select GEE users. "
            "Contact the agribound author for processing assistance."
        )
        break

## Cross-Year Stability Analysis

Evaluate how stable the field boundaries are from year to year using IoU and F1.

In [ ]:
if len(all_results) >= 2:
    print(f"{'Year':<6} {'Fields':>6} {'Total (ha)':>12} {'Avg (ha)':>10}")
    print(f"{'-' * 6} {'-' * 6} {'-' * 12} {'-' * 10}")
    for year, gdf in sorted(all_results.items()):
        area_ha = gdf["metrics:area"].sum() / 10000 if "metrics:area" in gdf.columns else 0
        avg_ha = gdf["metrics:area"].mean() / 10000 if "metrics:area" in gdf.columns else 0
        print(f"{year:<6} {len(gdf):>6} {area_ha:>12,.1f} {avg_ha:>10,.1f}")

    years_sorted = sorted(all_results.keys())
    for i in range(len(years_sorted) - 1):
        y1, y2 = years_sorted[i], years_sorted[i + 1]
        metrics = evaluate(all_results[y2], all_results[y1])
        print(
            f"\nStability {y1}\u2192{y2}: "
            f"IoU={metrics['iou_mean']:.3f} "
            f"F1={metrics['f1']:.3f} "
            f"P={metrics['precision']:.3f} "
            f"R={metrics['recall']:.3f}"
        )

## Visualization

In [ ]:
if all_results:
    from agribound.visualize import show_comparison

    m = show_comparison(
        list(all_results.values()),
        labels=[str(y) for y in sorted(all_results.keys())],
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_spot_timeseries.html"),
    )
    m